# Product Data Analysis Project

This notebook performs exploratory data analysis (EDA) on product-related data to identify pricing, rating, and bestseller trends using Python data libraries.

### Objectives:
- **Clean and Preprocess** raw Amazon product details
- **Conduct Statistical Analysis** on price distributions and ratings
- **Generate Modern Visualizations** with Matplotlib and Seaborn
- **Derive Key Business Insights** for marketing and product positioning

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set cohesive plotting styles
sns.set_theme(style="whitegrid", rc={
    "grid.color": "#e2e8f0",
    "grid.linestyle": "--",
    "axes.facecolor": "#f8fafc",
    "figure.facecolor": "#ffffff"
})
plt.rcParams.update({'font.size': 10, 'figure.dpi': 150})

## 1. Load and Inspect Raw Data

In [ ]:
# Load raw dataset
raw_path = "../amazon_product.csv" if os.path.exists("../amazon_product.csv") else "amazon_product.csv"
df = pd.read_csv(raw_path)
print(f"Raw dataset shape: {df.shape}")
df.head(3)

In [ ]:
# Inspect basic info
df.info()

## 2. Preprocessing & Data Cleaning

We clean the columns, retain critical features, parse price strings, and clean bestseller classifications.

In [ ]:
# Clean column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df.columns = [c.capitalize() for c in df.columns]

# Filter out duplicate rows
df.drop_duplicates(inplace=True)

# Keep only critical columns
cols_to_keep = ["Product_title", "Product_price", "Currency", "Product_star_rating", "Product_num_ratings", "Is_best_seller"]
df_cleaned = df[cols_to_keep].copy()

# Drop rows with missing values in key dimensions
df_cleaned.dropna(subset=cols_to_keep, inplace=True)

# Clean price strings and convert to float
df_cleaned["Product_price"] = df_cleaned["Product_price"].astype(str).str.replace("$", "", regex=False).str.strip().astype(float)

# Clean Bestseller boolean flag
df_cleaned["Is_best_seller"] = df_cleaned["Is_best_seller"].apply(lambda x: 1 if x is True or str(x).strip().lower() == 'true' else 0)

# Save the final cleaned dataframe
os.makedirs("data", exist_ok=True)
df_cleaned.to_csv("data/cleaned_product_data.csv", index=False)
print(f"Cleaned dataset shape: {df_cleaned.shape}")
df_cleaned.head(5)

## 3. Exploratory Data Analysis & Visualizations

Let's explore key distributions and correlations.

In [ ]:
# 3.1 Price Distribution
plt.figure(figsize=(7, 4))
sns.histplot(df_cleaned['Product_price'], bins=12, kde=True, color="#4f46e5")
plt.title("Distribution of Product Prices", fontsize=13, fontweight='bold', pad=10)
plt.xlabel("Price ($)")
plt.ylabel("Product Count")
plt.show()

In [ ]:
# 3.2 Product Rating Distribution
plt.figure(figsize=(7, 4))
sns.countplot(x='Product_star_rating', data=df_cleaned, hue='Product_star_rating', palette="viridis", legend=False)
plt.title("Distribution of Product Star Ratings", fontsize=13, fontweight='bold', pad=10)
plt.xlabel("Star Rating")
plt.ylabel("Number of Products")
plt.show()

In [ ]:
# 3.3 Bestseller Ratio count
plt.figure(figsize=(6, 4))
ax = sns.countplot(x='Is_best_seller', data=df_cleaned, hue='Is_best_seller', palette=['#10b981', '#f59e0b'], legend=False)
ax.set_xticklabels(["Non-Bestseller", "Bestseller"])
plt.title("Bestseller vs Non-Bestseller Counts", fontsize=13, fontweight='bold', pad=10)
plt.xlabel("Product Category")
plt.ylabel("Count")
plt.show()

In [ ]:
# 3.4 Ratings vs Engagement (Scatter)
plt.figure(figsize=(8, 4.5))
sns.scatterplot(
    x='Product_star_rating',
    y='Product_num_ratings',
    hue='Is_best_seller',
    style='Is_best_seller',
    palette={0: '#10b981', 1: '#f59e0b'},
    data=df_cleaned,
    s=120
)
plt.yscale("symlog")  # Adjust scale due to outlier counts
plt.title("Product Rating vs. Review Count", fontsize=13, fontweight='bold', pad=10)
plt.xlabel("Star Rating")
plt.ylabel("Number of Customer Reviews (Log Scale)")
plt.show()

In [ ]:
# 3.5 Average Price comparison
plt.figure(figsize=(6, 4))
sns.barplot(x='Is_best_seller', y='Product_price', data=df_cleaned, hue='Is_best_seller', palette=['#10b981', '#f59e0b'], errorbar=None, legend=False)
plt.gca().set_xticklabels(["Non-Bestseller", "Bestseller"])
plt.title("Average Product Price by Bestseller Status", fontsize=13, fontweight='bold', pad=10)
plt.xlabel("Product Category")
plt.ylabel("Average Price ($)")
plt.show()

In [ ]:
# 3.6 Correlation Heatmap
plt.figure(figsize=(5.5, 4.5))
corr = df_cleaned[['Product_price', 'Product_star_rating', 'Product_num_ratings']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".3f", linewidths=0.5)
plt.title("Correlation Analysis Heatmap", fontsize=13, fontweight='bold', pad=10)
plt.show()

In [ ]:
# 3.7 Top Reviewed Products
plt.figure(figsize=(9, 5))
top_product = df_cleaned.sort_values(by='Product_num_ratings', ascending=False).head(10).copy()
top_product['Short_title'] = top_product['Product_title'].str[:35] + '...'
sns.barplot(x='Product_num_ratings', y='Short_title', hue='Is_best_seller', palette={0: '#10b981', 1: '#f59e0b'}, data=top_product)
plt.title("Top 10 Most Reviewed Products", fontsize=13, fontweight='bold', pad=10)
plt.xlabel("Number of Customer Reviews")
plt.ylabel("Product Title")
plt.show()